# Which SnC snapshots?

`_handle_snapshot_creation` fills its budget greedily, so a 50-epoch run takes
both snapshots at the ends of epochs 1 and 2. By then the cubic schedule has
applied **0% and 8%** of the target sparsity, so SnC spends the remaining 48
epochs pulling the student toward two nearly-dense copies of itself.

`--snapshot_schedule even` spreads the same budget across the pruning phase
instead:

| policy | snapshot epochs | sparsity applied by then |
|---|---|---|
| `first` n=2 (current) | 1, 2 | 0%, 8% |
| `even` n=2 | 17, 33 | 78%, 99% |
| `even` n=3 | 12, 25, 38 | 63%, 95%, 100% |
| `even` n=5 | 8, 17, 25, 33, 42 | 49%, 78%, 95%, 99%, 100% |

Eight runs: four policies x two sparsities, resnet34/magnitude, seed 1.
0.97 is where SnC has little to preserve; 0.999 is where the anchor should
matter most. If the policies separate anywhere it should be at 0.999.

More snapshots cost a forward pass per batch each, so `even5` runs roughly 1.5x
longer than `first2`. That is part of the answer, not an artifact.


In [ ]:
import sys, pathlib

here = pathlib.Path.cwd()
while not (here / '.git').exists() and here != here.parent:
    here = here.parent
sys.path.insert(0, str(here / 'project' / 'test_notebooks'))

import nb_common as nb
info = nb.setup()


## Plan

In [ ]:
MODEL      = 'resnet34'
PRUNER     = 'magnitude'
SPARSITIES = (0.97, 0.999)
SEED       = 1
GPU        = 0

# (variant tag, num_snapshots, schedule)
ARMS = [
    ('snapfirst2', 2, 'first'),   # control: what every run so far did
    ('snapeven2',  2, 'even'),
    ('snapeven3',  3, 'even'),
    ('snapeven5',  5, 'even'),
]

plan = []
for tag, n, sched in ARMS:
    for sp in SPARSITIES:
        plan.append(nb.make_cell(MODEL, 'bacp', seed=SEED, pruner=PRUNER, sparsity=sp,
                                 variant=tag, num_snapshots=n, snapshot_schedule=sched))

# the snapshot policy must be the ONLY thing that varies
ref = nb.FAMILIES[MODEL]['bacp']
for c in plan:
    for k in ('learning_rate', 'epochs', 'epochs_ft', 'delta_T', 'sparsity_scheduler',
              'recovery_epochs', 'val_split', 'prune_task_head', 'wanda_group',
              'optimizer_type', 'batch_size', 'num_classes', 'dataset_name', 'tau',
              'contrastive_mode', 'proj_mode', 'lambdas'):
        if k in ref:
            assert c['config'][k] == ref[k], (c['key'], k, c['config'][k], ref[k])
    assert c['config']['epochs'] == 50 and c['config']['epochs_ft'] == 25

by_tag = {}
for (tag, n, sched) in ARMS:
    got = [c for c in plan if c['key'].endswith('.' + tag)]
    assert len(got) == len(SPARSITIES), tag
    for c in got:
        assert c['config']['num_snapshots'] == n
        assert c['config']['snapshot_schedule'] == sched
    by_tag[tag] = (n, sched)

print('%d runs' % len(plan))
for tag, (n, sched) in by_tag.items():
    eps = sorted({round(50 * (i + 1) / (n + 1)) for i in range(n)}) if sched == 'even' \
        else list(range(1, n + 1))
    print('  %-11s n=%d %-6s snapshots at epochs %s' % (tag, n, sched, eps))
print('est ~%.0f min' % sum(8.1 * (1.0 if c['config']['num_snapshots'] <= 2 else
                                   1.0 + 0.16 * (c['config']['num_snapshots'] - 2))
                            for c in plan))
assert nb.sanity_check(plan), 'sanity check failed'


## Run

Control first, so a partial run still has something to compare against.

In [ ]:
nb.run_group(plan, gpu=GPU)


## Verdict

In [ ]:
import json, glob, os

root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok':
        acc[k] = (r.get('test_acc_exact_pct') or r.get('test_acc_pct'),
                  (r.get('duration_s') or 0) / 60)

print('%-12s %10s %10s   %10s %10s' % ('policy', '0.97', 'min', '0.999', 'min'))
print('-' * 58)
base = {}
for tag, n, sched in ARMS:
    row = []
    for sp in SPARSITIES:
        v = acc.get('static.bacp.%s.cifar10.s%s.%s.seed%d.%s'
                    % (MODEL, sp, PRUNER, SEED, tag))
        row.append(v)
        if tag == 'snapfirst2' and v:
            base[sp] = v[0]
    cells = []
    for sp, v in zip(SPARSITIES, row):
        if v is None:
            cells += ['     --   ', '     --   ']
        else:
            d = ('' if tag == 'snapfirst2' or sp not in base
                 else ' (%+.2f)' % (v[0] - base[sp]))
            cells += ['%6.2f%s' % (v[0], d), '%9.1f' % v[1]]
    print('%-12s %10s %10s   %10s %10s' % (tag, *cells))
print()
print('(+/-) is against snapfirst2, the current default, at the same sparsity.')
print('One seed: read the direction, not the decimal.')
